# Practice 1 — Personal Research Assistant

This notebook builds the capstone project from `8_Practice/README.md` cell by cell.

**Concepts practised:**
- LLM setup with `langchain_openai` / `langchain_groq`
- Tool definition and function calling
- Simple ReAct agent
- Pydantic structured output
- Embeddings and FAISS vector search
- Full RAG pipeline
- Putting everything together

**Run each cell, read the markdown, and experiment.**

In [ ]:
from dotenv import load_dotenv
import os
import warnings
warnings.filterwarnings("ignore")

load_dotenv(override=True)

# Safe fallback: if a key is missing the cell still runs.
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN", "")

print("Environment loaded. Missing keys will show empty strings below:")
for key in ["OPENAI_API_KEY", "GROQ_API_KEY", "HF_TOKEN"]:
    value = os.environ.get(key, "")
    print(f"  {key}: {'set' if value else 'NOT SET'}")

## Step 1 — Choose an LLM

We use `ChatOpenAI` if `OPENAI_API_KEY` is available, otherwise fall back to `ChatGroq` (which has a free tier).

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

if os.environ.get("OPENAI_API_KEY"):
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
    print("Using OpenAI gpt-4o-mini")
elif os.environ.get("GROQ_API_KEY"):
    llm = ChatGroq(model="llama3-8b-8192", temperature=0.3)
    print("Using Groq llama3-8b-8192")
else:
    raise ValueError("No API key found. Set OPENAI_API_KEY or GROQ_API_KEY in your .env file.")

# Quick sanity check
response = llm.invoke("Say 'LLM is ready' in one sentence.")
print(response.content)

## Step 2 — Build Tools

Tools are just Python functions wrapped with a name and description. The agent uses the description to decide when to call each tool.

In [ ]:
from langchain_core.tools import Tool
from langchain_community.tools import DuckDuckGoSearchResults

# 1. Calculator tool
def calculator(expression: str) -> str:
    '''Evaluate a simple math expression like '15 * 23'.'''
    try:
        # eval is acceptable here because YOU control the input in a practice notebook.
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

# 2. Date tool
def today() -> str:
    '''Return today's date.'''
    from datetime import date
    return str(date.today())

# 3. Web search tool (no API key needed)
search_tool = DuckDuckGoSearchResults(max_results=5)

tools = [
    Tool(name="calculator", func=calculator, description="Use for math expressions."),
    Tool(name="today", func=today, description="Use when you need today's date."),
    Tool(name="web_search", func=search_tool.run, description="Use for current information from the web.")
]

print(f"Loaded {len(tools)} tools:")
for t in tools:
    print(f"  - {t.name}: {t.description}")

## Step 3 — Create a Tool-Using Agent

We create a ReAct-style agent. The agent loops:
1. Reads the user question
2. Decides which tool to call (if any)
3. Executes the tool
4. Writes the final answer

In [ ]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain.prompts import hub

# Load a built-in ReAct prompt template
prompt = hub.pull("hwchase17/react")

# Create the agent
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

# Test: math + current info
result = agent_executor.invoke({
    "input": "What is 15% of 24000 and what is one trending topic in AI today?"
})
print("\n--- FINAL ANSWER ---")
print(result["output"])

## Step 4 — Structured Output with Pydantic

Instead of free text, force the LLM to return a typed object. This is essential for chaining outputs into other functions.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class ResearchFact(BaseModel):
    claim: str = Field(description="A single factual statement")
    source: str = Field(description="URL or source of the claim")
    confidence: str = Field(description="High, Medium, or Low")

class ResearchReport(BaseModel):
    question: str = Field(description="Original research question")
    summary: str = Field(description="Short summary of findings")
    facts: List[ResearchFact] = Field(description="List of extracted facts")

# Create a chain that always returns ResearchReport
from langchain_core.prompts import ChatPromptTemplate

structure_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a research assistant. Extract structured facts from the context."),
    ("human", "Question: {question}\n\nContext:\n{context}")
])

structured_chain = structure_prompt | llm.with_structured_output(ResearchReport)

# Test with synthetic context
sample = structured_chain.invoke({
    "question": "What is FAISS?",
    "context": "FAISS is a library from Meta for efficient similarity search of dense vectors."
})
print(sample)

## Step 5 — RAG with FAISS

Load a web page, split it, embed it, store it in FAISS, and retrieve relevant chunks.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Load a public article (change the URL to anything you like)
url = "https://python.langchain.com/docs/introduction/"
loader = WebBaseLoader(url)
docs = loader.load()
print(f"Loaded {len(docs)} document(s) from {url}")

# 2. Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(f"Created {len(chunks)} chunks")

# 3. Embed with a local sentence-transformer model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 4. Store in FAISS
vector_store = FAISS.from_documents(chunks, embeddings)
print("Vector store ready.")

# 5. Retrieve
query = "What is LangChain used for?"
retrieved = vector_store.similarity_search(query, k=3)

print(f"\nTop 3 chunks for: '{query}'\n")
for i, chunk in enumerate(retrieved, 1):
    print(f"--- Chunk {i} ---")
    print(chunk.page_content[:300])
    print()

## Step 6 — Put It All Together

Combine search, RAG, and structured output into one `research_assistant` function.

In [ ]:
def research_assistant(question: str) -> ResearchReport:
    '''Search the web, retrieve relevant chunks, and return a structured report.'''
    # 1. Web search
    print("Searching the web...")
    search_results = search_tool.run(question)
    print(search_results[:500])

    # 2. Use the search results as our 'context' for this demo
    #    (In a full app you would parse URLs and load them with WebBaseLoader.)
    context = search_results

    # 3. Generate structured report
    print("Generating structured report...")
    return structured_chain.invoke({
        "question": question,
        "context": context
    })

# Run it
question = "What are the latest trends in generative AI in 2025?"
report = research_assistant(question)

print("\n--- STRUCTURED REPORT ---")
print(f"Question: {report.question}")
print(f"Summary: {report.summary}")
print("\nFacts:")
for f in report.facts:
    print(f"  • [{f.confidence}] {f.claim} (source: {f.source})")

## Step 7 (Optional) — RAG on Your Own Documents

Try loading a local `.txt` or `.pdf` file and answering questions from it.

In [ ]:
# Example: load a local text file
# from langchain_community.document_loaders import TextLoader

# loader = TextLoader("../study_notes/10_rag_fundamentals.md")
# my_docs = loader.load()
# my_chunks = splitter.split_documents(my_docs)
# my_store = FAISS.from_documents(my_chunks, embeddings)

# answer = my_store.similarity_search("What are the six steps of RAG?", k=2)
# for a in answer:
#     print(a.page_content)

print("Uncomment and adapt the code above to practice RAG on your own files.")

## Exercises

1. Add a `unit_converter` tool (km ↔ miles, kg ↔ lbs).
2. Change the LLM temperature and compare outputs.
3. Increase `chunk_size` to 1000 and see if retrieval quality changes.
4. Replace DuckDuckGo search with `TavilySearchResults` (needs `TAVILY_API_KEY`).
5. Build a second agent that fact-checks the `ResearchReport` and returns a critique.
6. Save and load the FAISS index with `vector_store.save_local()` and `FAISS.load_local()`.
7. Create a notebook `02_my_own_assistant.ipynb` that solves a problem you care about.